Price_Optimization

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [5]:
df = pd.read_csv("../data/processed/feature_engineered_sales_data.csv")
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Month,Day,DayOfWeek,Quarter,Price_Difference,Price_Ratio,Price_Difference_Percent,Effective_Price,Stock_Status,Inventory_to_Order_Ratio
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,1,1,5,1,-13.01,0.848244,-15.175551,69.084,In_Stock,0.770751
1,2022-01-01,S004,P0013,Groceries,West,136,104,385,20.24,10,...,1,1,5,1,1.34,1.070899,7.089947,18.216,In_Stock,0.352332
2,2022-01-01,S004,P0012,Electronics,West,111,111,113,118.15,0,...,1,1,5,1,-15.31,0.885284,-11.471602,118.150,In_Stock,0.973684
3,2022-01-01,S004,P0011,Clothing,West,195,60,293,52.89,0,...,1,1,5,1,-9.40,0.849093,-15.090705,52.890,In_Stock,0.663265
4,2022-01-01,S004,P0010,Groceries,West,223,120,597,30.02,0,...,1,1,5,1,0.87,1.029846,2.984563,30.020,In_Stock,0.372910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75995,2024-01-30,S002,P0008,Groceries,South,85,81,94,26.98,0,...,1,30,1,1,2.33,1.094523,9.452333,26.980,In_Stock,0.894737
75996,2024-01-30,S002,P0007,Toys,South,414,126,0,26.84,10,...,1,30,1,1,-2.71,0.908291,-9.170897,24.156,In_Stock,414.000000
75997,2024-01-30,S002,P0006,Clothing,South,98,96,117,63.58,10,...,1,30,1,1,-12.64,0.834164,-16.583574,57.222,In_Stock,0.830508
75998,2024-01-30,S002,P0016,Clothing,South,88,78,0,91.50,5,...,1,30,1,1,5.76,1.067180,6.717985,86.925,In_Stock,88.000000


In [6]:
import joblib

final_model = joblib.load(
    "../models/final_demand_model.pkl"
)

In [7]:
base_data = df.iloc[0].copy()

base_data

Date                         2022-01-01
Store ID                           S001
Product ID                        P0001
Category                    Electronics
Region                            North
Inventory Level                     195
Units Sold                          102
Units Ordered                       252
Price                             72.72
Discount                              5
Weather Condition                 Snowy
Promotion                             0
Competitor Pricing                85.73
Seasonality                      Winter
Epidemic                              0
Demand                              115
Year                               2022
Month                                 1
Day                                   1
DayOfWeek                             5
Quarter                               1
Price_Difference                 -13.01
Price_Ratio                    0.848244
Price_Difference_Percent     -15.175551
Effective_Price                  69.084


In [8]:
current_price = base_data["Price"]

price_range = np.linspace(
    current_price * 0.80,
    current_price * 1.20,
    21
)

price_range

array([58.176 , 59.6304, 61.0848, 62.5392, 63.9936, 65.448 , 66.9024,
       68.3568, 69.8112, 71.2656, 72.72  , 74.1744, 75.6288, 77.0832,
       78.5376, 79.992 , 81.4464, 82.9008, 84.3552, 85.8096, 87.264 ])

In [9]:
price_scenarios = pd.DataFrame(
    [base_data] * len(price_range)
)

price_scenarios["Price"] = price_range

In [10]:
price_scenarios["Price_Difference"] = (
    price_scenarios["Price"]
    - price_scenarios["Competitor Pricing"]
)

price_scenarios["Price_Difference_Percent"] = (
    price_scenarios["Price_Difference"]
    / price_scenarios["Competitor Pricing"]
) * 100

price_scenarios["Effective_Price"] = (
    price_scenarios["Price"]
    * (1 - price_scenarios["Discount"] / 100)
)

In [11]:
predicted_demand = final_model.predict(
    price_scenarios
)

price_scenarios["Predicted_Demand"] = predicted_demand

In [12]:
price_scenarios["Expected_Revenue"] = (
    price_scenarios["Price"]
    * price_scenarios["Predicted_Demand"])

In [13]:
optimal_row = price_scenarios.loc[
    price_scenarios["Expected_Revenue"].idxmax()
]

optimal_price = optimal_row["Price"]
optimal_demand = optimal_row["Predicted_Demand"]
optimal_revenue = optimal_row["Expected_Revenue"]

print("Optimal Price:", optimal_price)
print("Predicted Demand:", optimal_demand)
print("Expected Revenue:", optimal_revenue)

Optimal Price: 0    58.1760
0    59.6304
0    61.0848
0    62.5392
0    63.9936
0    65.4480
0    66.9024
0    68.3568
0    69.8112
0    71.2656
0    72.7200
0    74.1744
0    75.6288
0    77.0832
0    78.5376
0    79.9920
0    81.4464
0    82.9008
0    84.3552
0    85.8096
0    87.2640
Name: Price, dtype: float64
Predicted Demand: 0    109.451150
0    108.111650
0    108.111650
0    107.080222
0    106.613841
0    106.903007
0    108.465674
0    109.151627
0    109.031805
0    109.278139
0    111.060759
0    112.329532
0    112.894090
0    114.351470
0    113.706133
0    115.162243
0    115.745528
0    115.214397
0    115.143683
0    113.869633
0    113.403305
Name: Predicted_Demand, dtype: float64
Expected Revenue: 0    6367.430122
0    6446.740955
0    6603.978539
0    6696.711405
0    6822.603484
0    6996.588034
0    7256.613918
0    7461.255905
0    7611.641176
0    7787.772126
0    8076.338379
0    8331.975602
0    8538.044550
0    8814.577194
0    8930.206826
0    9212.058118
0

In [14]:
price_scenarios[
    [
        "Price",
        "Predicted_Demand",
        "Expected_Revenue"
    ]
].sort_values(
    "Expected_Revenue",
    ascending=False
).head(10)

,Price,Predicted_Demand,Expected_Revenue
0,87.2640,113.403305,9896.026026
0,85.8096,113.869633,9771.107703
0,84.3552,115.143683,9712.968423
0,82.9008,115.214397,9551.365721
0,81.4464,115.745528,9427.056605
0,79.9920,115.162243,9212.058118
0,78.5376,113.706133,8930.206826
0,77.0832,114.351470,8814.577194
0,75.6288,112.894090,8538.044550
0,74.1744,112.329532,8331.975602
